# Week 3 (together), Modelling: logistic regression, built by the room

**Today's group work is the modelling, and modelling is a sequence of choices.** What the two
piles are. What counts as a feature. What score you would actually believe. Which of several
models you keep, and why. If the code were already written you would read those choices
instead of making them, so the stations below are blank.

**Blank is not unguided.** Every station gives you the concept in plain language, the exact
tools to reach for, and a way to tell whether what came back is right. What it doesn't give you
is the code. That part is yours and the AI's.

**What your group leaves with:** a number you can defend, two word lists you can interpret, a
model you broke on purpose, and one caveat you would put in writing.

---

### How to work

Threes, one screen, and rotate these three jobs at every station:

| Job | Does |
|---|---|
| **Driver** | Types, and prompts the AI. Never types a line nobody has read aloud. |
| **Reader** | Says what the cell will do *before* it runs, then whether it did. |
| **Skeptic** | Asks the awkward question. Is that better than guessing? Would that word survive on someone else's data? |

### The stations, and roughly what they cost

| # | Station | Time |
|---|---|---|
| 0 | Warm-up: the whole pipeline on six toy sentences (written for you) | 3 min |
| 1 | The question, and your prediction | 3 min |
| 2 | Features: turn text into numbers | 4 min |
| 3 | Fit the model | 4 min |
| 4 | Judge it honestly | 5 min |
| 5 | Read its mind | 5 min |
| 6 | Break it | 4 min |
| 7 | Change one thing and fit again — **the point of the session** | 6 min |
| 8 | Report back | 3 min |

Behind schedule? Stations 4, 5, and 7 are the ones that must happen. The worked version of
this pipeline is `week03_classification.ipynb` — peek if you're stuck, after you've asked the
AI and read what it gave you.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.

In [ ]:
# If an import fails: re-run this cell; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, models, and charts are
# still there next week. (Outside Colab - e.g. the offline test harness - this falls
# back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("imports ok")

---

## Station 0 · The whole pipeline, on six sentences (3 min, written for you)

Before you build anything on real data, watch the entire method happen on a corpus small enough
to hold in your head: six sentences, three about the sea and three about the kitchen.

Run the next cell, then read it line by line. **Everything you do for the rest of the session is
this cell, at scale, with your own choices substituted.** The four moving parts, in order:

1. **Features.** `CountVectorizer` builds a vocabulary from the corpus and turns each document
   into a row of counts. Rows are documents, columns are words.
2. **Weights.** `LogisticRegression` learns one number per column — positive pushes toward one
   label, negative toward the other, near-zero means the word carried no information.
3. **The fit.** Fitting *is* the search for those weights: the set that leans the right way for
   as many training documents as possible.
4. **Reading it.** `model.coef_` is the weights, `vectorizer.get_feature_names_out()` is the
   words. Line them up, sort, and you are reading the model's mind.

Predict before you run: which words will get the biggest weights?

In [ ]:
# --- The toy corpus: six documents, two labels --------------------------------
toy = pd.DataFrame({
    "text": ["the tide came in over the cold sand",
             "salt spray and a grey sea all morning",
             "we walked the shore until the tide turned",
             "he chopped onions and let the butter brown",
             "the bread proved overnight on the warm counter",
             "she salted the water and dropped the pasta in"],
    "label": ["sea", "sea", "sea", "kitchen", "kitchen", "kitchen"],
})

# 1. FEATURES: text -> a matrix of word counts.
toy_vec = CountVectorizer()
Xt = toy_vec.fit_transform(toy["text"])       # rows = documents, columns = words
yt = (toy["label"] == "sea").astype(int)      # the target: 1 = sea, 0 = kitchen
print("matrix shape (documents, vocabulary):", Xt.shape)
print("first ten of the vocabulary:", list(toy_vec.get_feature_names_out()[:10]))

# 2 + 3. WEIGHTS + FIT: one number per word, learned from the labeled examples.
toy_clf = LogisticRegression(max_iter=1000).fit(Xt, yt)

# 4. READ IT: line the words up against their weights and sort.
toy_words = np.array(toy_vec.get_feature_names_out())
toy_w = toy_clf.coef_.ravel()
order = toy_w.argsort()
print("\nmost KITCHEN:", ", ".join(toy_words[order[:4]]))
print("most SEA:     ", ", ".join(toy_words[order[-4:][::-1]]))

# Four weights worth looking at by name (the markdown below says why):
for word in ["in", "the", "salt", "salted"]:
    print(f"  weight of {word!r:9}", round(toy_w[list(toy_words).index(word)], 3))
print("accuracy on its OWN training rows:", toy_clf.score(Xt, yt), "<- meaningless")

# And the model as a reader: hand it a sentence it has never seen.
for s in ["the salt water boiled", "the grey morning sand"]:
    p = toy_clf.predict_proba(toy_vec.transform([s]))[0, 1]
    print(f"{s!r:28} -> {'sea' if p > 0.5 else 'kitchen'} (p_sea={p:.2f})")

**Read the output before you move on.** Three things worth noticing, because all three come
back at full scale:

- **Uninformative words earn small weights on their own.** Look up *in*: it appears once on
  each side, and its weight is essentially zero. The model needs no stop list to ignore it.
  But look up *the* as well — it lands a small non-zero weight, because in six documents it
  happens to fall slightly more on one side. With this little evidence, noise gets a weight
  too, and that never entirely stops being true.
- **The model has no idea that *salt* and *salted* are the same word.** They're separate
  columns here, pushing in opposite directions, one from the sea sentences and one from the
  kitchen. That's the exact run/running argument you had by hand in Week 2, now visible as two
  numbers.
- **Its accuracy on its own training rows is 100 percent, and means nothing.** Six documents,
  thirty-six columns: it can memorise. Nothing has been tested on data it didn't see. Station 4
  is where that gets fixed, and it's the single most common way a real result goes wrong.

One vocabulary note for the stations ahead: **features** are the columns (here, words),
**weights** (or coefficients) are the learned numbers, **fitting** or **training** is the
search for them, and **held-out** data is rows the model never saw during that search.

---

## Your corpus (written for you): two labelled piles

A classifier needs examples someone has already sorted. `load_pair()` hands your group a
DataFrame with two columns, `text` and `label`, and nothing else. Collecting a corpus is Week
4's whole subject; today it's plumbing.

**Your one setup choice: which two piles.** Set `PAIR` in the next cell.

| Pair | What it teaches |
|---|---|
| `("sandiego", "SanDiegan")` | The hard default. Two communities about one city — same beaches, same rents, largely the same words. Expect an accuracy near 0.6 and a genuinely interesting weight list. |
| Any two subreddits you know | `("coffee", "espresso")`, `("AskMen", "AskWomen")`, `("nba", "soccer")`. The archive covers essentially all of Reddit; comments are pulled live and analyzed only, never redistributed. |
| `("shelley", "stoker")` | Sentences from *Frankenstein* and *Dracula*. No network needed, and the automatic fallback if the archive is slow or a whole classroom hits it at once. |

**Choose deliberately.** Two unrelated topics gets you 95 percent accuracy and a boring
reading; the words are just the two subjects. A hard pair gets you 60 percent, and the words are
a finding. A model that barely beats a coin flip is a *result* — it says these two communities
write alike — as long as you can show the baseline it beat.

In [ ]:
# --- Provided loader. Read it, don't rewrite it. ------------------------------
PAIR = ("sandiego", "SanDiegan")   # <-- your group's two piles (see the table above)
N_PER_SIDE = 400                   # rows per pile; 400 trains in a couple of seconds

def load_reddit(a, b, n=N_PER_SIDE):
    """Two subreddits via the Arctic Shift archive API (analyze-only).
    Retries politely: a whole classroom at once gets rate-limited."""
    import requests, time
    def pull(sub, tries=2):
        for attempt in range(tries):
            got, before, pages = [], None, 0
            try:
                while len(got) < n and pages < 10:
                    params = {"subreddit": sub, "limit": 100, "fields": "body,created_utc"}
                    if before: params["before"] = before
                    resp = requests.get("https://arctic-shift.photon-reddit.com/api/comments/search",
                                        params=params, timeout=30)
                    resp.raise_for_status()
                    rows = resp.json()["data"]
                    pages += 1
                    if not rows: break
                    before = int(min(r["created_utc"] for r in rows))
                    got += [r["body"] for r in rows if isinstance(r.get("body"), str)
                            and 80 < len(r["body"]) < 800
                            and "[removed]" not in r["body"] and "[deleted]" not in r["body"]
                            and "moderator" not in r["body"].lower()
                            and "has been removed" not in r["body"].lower()]
                if got:
                    return got[:n]
            except Exception as e:
                print(f"r/{sub}: {type(e).__name__} (attempt {attempt + 1}/{tries}), waiting...")
            time.sleep(3 * (attempt + 1))
        return []
    a_txt, b_txt = pull(a), pull(b)
    if not (a_txt and b_txt): raise ValueError("empty pull")
    return pd.DataFrame({"text": a_txt + b_txt, "label": [a] * len(a_txt) + [b] * len(b_txt)})

def load_novelists(n=N_PER_SIDE):
    """Shelley against Stoker: the repo's snapshots first, Gutenberg if they aren't there."""
    def sents(text):
        return [s.strip().replace("\n", " ") for s in re.split(r"(?<=[.!?])\s+", text)
                if 40 < len(s) < 180][:n]
    def read_one(fname, url):
        for base in ("data/texts", "notebooks/data/texts", "../notebooks/data/texts"):
            path = os.path.join(base, fname)
            if os.path.exists(path):
                return open(path, encoding="utf-8", errors="ignore").read()
        import requests
        raw = requests.get(url, timeout=30).text.replace("\r\n", "\n")
        body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
        return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]
    a = sents(read_one("frankenstein.txt", "https://www.gutenberg.org/cache/epub/84/pg84.txt"))
    b = sents(read_one("dracula.txt", "https://www.gutenberg.org/cache/epub/345/pg345.txt"))
    return pd.DataFrame({"text": a + b, "label": ["shelley"] * len(a) + ["stoker"] * len(b)})

def load_pair(pair=PAIR):
    """Returns (df, label_a, label_b). Falls back to the novelists if the archive is down."""
    a, b = pair
    if {a, b} == {"shelley", "stoker"}:
        return load_novelists(), "shelley", "stoker"
    try:
        return load_reddit(a, b), a, b
    except Exception as e:
        print("archive unavailable:", type(e).__name__, "- falling back to the novelists")
        return load_novelists(), "shelley", "stoker"

df, LABEL_A, LABEL_B = load_pair()
print(df["label"].value_counts().to_string())
df.sample(4, random_state=1)

**Before you model, look.** Two numbers and one habit:

- How many rows per side? If one pile is much bigger, remember that number — it comes back at
  Station 4 as the score a lazy model gets for free.
- Read two comments aloud, in full. Everything you can only learn by looking, you learn now.
  Half the surprises at Station 5 are things a human would have spotted in thirty seconds of
  reading (a bot posting the same template, a pinned thread, one loud user).

### Asking the AI well

The AI writes today's code; you make today's decisions. Prompts that work are small and
concrete, and name the objects you already have:

> *"Using the DataFrame `df` with columns text and label, vectorize the text with
> CountVectorizer and show me the matrix shape."*

> *"Split that into train and test with 25 percent held out, stratified by y, random_state 0.
> Then fit a LogisticRegression and print the accuracy on the held-out part."*

> *"What does min_df=3 change about the matrix, in one sentence?"*

Prompts that don't: *"do week 3 for me."* You get code you can't defend at report-back, and the
Reader has nothing to narrate. Read every line back before running it — that's the job.

---

## Station 1 · The question, and your prediction (3 min)

One sentence, written down: what would it *mean* if a machine could tell these two piles apart,
and what would it mean if it couldn't? "Can a classifier do it" is not yet a question about
culture; "do these two communities write differently enough that a machine can hear it" is.

Then predict, out loud and in writing: what accuracy do you expect, and three words you think
will give each side away. A prediction made after seeing the result is not a prediction — and
being wrong here is the most useful thing that can happen to you today, because a surprise is
the only reliable sign you've learned something about the corpus rather than about sklearn.

In [ ]:
# OUR QUESTION: ...one sentence...
# OUR PREDICTION: accuracy about ..., because ...
# GIVEAWAY WORDS WE EXPECT: side A: ..., ..., ...   side B: ..., ..., ...

# Optional, and worth the thirty seconds: print a couple of full-length rows from
# each pile so the prediction is made with the text in front of you.

---

## Station 2 · Features: turn text into numbers (4 min)

A logistic regression cannot see text. Something has to turn each document into a row of
numbers, and that something is your first modelling choice — made, if you don't make it,
by the defaults.

`CountVectorizer` builds a vocabulary from your corpus and counts. Its defaults quietly decide
a great deal: it lowercases everything (so *Reddit* and *reddit* are one column), splits on a
pattern that drops punctuation and single characters, keeps every word that survives, and
treats *run* and *running* as unrelated columns. Every one of those was a decision Week 2 made
you argue about by hand.

**Build it.** Turn `df["text"]` into a matrix `X`, and `df["label"]` into a 0/1 target `y`.
Then print the shape and a slice of the vocabulary.

**Tools:** `CountVectorizer()`, `.fit_transform(...)`, `.get_feature_names_out()`, and
`(df["label"] == LABEL_A).astype(int)` for the target.

**Check yourself.** The shape should be (about 800, several thousand). If the column count is
in the tens of thousands you have a big vocabulary of near-unique words — fine, and Station 7's
`min_df` dial is about exactly that. If the row count isn't your corpus size, something went
wrong before you modelled anything.

**Discuss (Skeptic's question):** how many columns did you get, and which of those defaults
produced that number? Name one you'd change if this were your project.

In [ ]:
# 1. Vectorize df["text"] into X.
# 2. Build y: 1 for LABEL_A, 0 for LABEL_B.
# 3. Print X.shape, and the first 20 feature names.
#
# HOW MANY COLUMNS: ...
# THE DEFAULT WE'D CHANGE, AND WHY: ...

---

## Station 3 · Fit the model (4 min)

Before you run it, the Reader says what fitting does: **the model is searching for one weight
per word, such that the weighted sum of a document's words leans the right way for as many
training documents as possible.** No understanding, no rules anyone wrote — a few thousand
numbers, tuned until the votes come out right.

Two things to get right here, because everything downstream depends on them:

- **Hold data out.** `train_test_split` with `test_size=0.25` keeps a quarter of the rows away
  from training so you have something honest to score on. Use `stratify=y` so both piles are
  represented in both halves, and `random_state=0` so your Station 7 comparisons are against
  the same split rather than a different roll of the dice.
- **Let it converge.** `LogisticRegression(max_iter=1000)`. The default iteration cap is low
  for text-sized vocabularies, and the warning you'd otherwise get means "the optimiser ran out
  of steps", not "your model is wrong".

**Check yourself.** Print the held-out accuracy. Anything from about 0.55 (a hard pair) to 0.99
(an easy one) is plausible. Exactly 1.00 is a red flag, not a triumph: something in the text is
giving the label away — a subreddit name inside its own comments, a template, a scraping
artifact. That's called leakage, and finding it is a better result than the 1.00 was.

In [ ]:
# 1. Split: Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
# 2. Fit a LogisticRegression(max_iter=1000) on the training half.
# 3. Print the accuracy on the held-out half.
# 4. Refit a second copy on ALL the rows - that's the one to read weights from at
#    Station 5, since it has seen the most evidence. Keep the two straight.
#
# HELD-OUT ACCURACY: ...
# ANYTHING SUSPICIOUS ABOUT IT: ...

---

## Station 4 · Judge it honestly (5 min)

An accuracy number on its own means nothing. Three things give it meaning, and this station is
where most published mistakes would have been caught.

**1. A baseline.** What does a model that always guesses the bigger pile score? If your piles
are 400/364, that's 52 percent for free, without reading a word. Your model's number is only
interesting as a *distance above* that floor. `DummyClassifier(strategy="most_frequent")` fits
and scores exactly like a real model, which is the point — it's the null hypothesis you can run.

**2. The held-out rule.** Score on rows the model never trained on. Score on training rows and
you're measuring memory: with thousands of word-columns and hundreds of documents, a logistic
regression can nearly memorise the training set, and it will happily tell you 98 percent.

**3. The shape of the errors.** A confusion matrix says *which* side it gets wrong: rows are
the true labels, columns are the predictions, and the off-diagonal cells are the mistakes. A
model that's 60 percent accurate by calling almost everything pile A is a different animal from
one that's 60 percent accurate evenly. `classification_report` says the same thing with
precision and recall attached.

**Optional if you're quick:** `cross_val_score(clf, X, y, cv=5)` refits on five different
splits. The spread tells you how much your single accuracy number was luck — often ±3 points,
which is worth knowing before you defend a 2-point difference at Station 7.

**Discuss:** how much better than the baseline are you, and is that gap the finding, or is the
gap so small that "these two write alike" is the finding?

In [ ]:
# 1. Baseline: fit DummyClassifier(strategy="most_frequent") on the same split, score it.
# 2. Your model's held-out accuracy, printed next to it.
# 3. confusion_matrix(yte, predictions) - and say out loud which cell is which.
# 4. Optional: cross_val_score(..., cv=5) and its mean and standard deviation.
#
# BASELINE: ...    OUR MODEL: ...    DISTANCE ABOVE THE FLOOR: ...
# WHICH SIDE IT GETS WRONG: ...

---

## Station 5 · Read its mind (5 min)

This is why today's model is a logistic regression and not something cleverer. Every word has
one signed weight; positive pushes toward one label, negative toward the other, and the largest
of each are the model's reasoning laid out in full. Nothing in Week 7's annotator will let you
do this.

**Get the two arrays and line them up.** `model.coef_.ravel()` is the weights,
`vectorizer.get_feature_names_out()` is the words, in the same column order.
`weights.argsort()` gives you the indices from most negative to most positive — take from both
ends. A horizontal bar chart of the top six each way (`plt.barh`) makes it a slide.

**Then do the part that's actually the lesson: sort the top words into three kinds.**

| Kind | Means | Example |
|---|---|---|
| **Topic** | the two piles talk about different things | *beaches*, *lebron*, *espresso* |
| **Register** | the two piles talk in different styles | *citation*, *lol*, *therefore* |
| **Community habit** | the two piles have different rituals | *this sub*, *mods*, *OP*, *edit* |

A model living on topic tells you the communities discuss different subjects — often obvious
before you started. A model living on register or habit is the more interesting claim: same
subject, different voice. **Which kind is yours, and does that answer the question you wrote at
Station 1?**

One caution the Skeptic should raise: a weight is not evidence of frequency. A rare word that
appears in four documents, all on one side, can earn a large weight. Before you build a story
on a word, check how often it actually occurs — `X[:, index].sum()`, or just search the text.

In [ ]:
# 1. words = vectorizer.get_feature_names_out(); w = model.coef_.ravel()
# 2. Top ten each way, printed with the label they push toward.
# 3. Optional: plt.barh of the top six each way.
# 4. Spot-check one word you find surprising: how many documents is it actually in?
#
# TOP WORDS, SIDE A: ...
# TOP WORDS, SIDE B: ...
# TOPIC / REGISTER / HABIT - our sort: ...
# THE ONE THAT SURPRISED US: ...

---

## Station 6 · Break it (4 min)

Your classifier has exactly two boxes and no concept of *neither*. Hand it a line of
Shakespeare, a recipe, a sentence in another language, and it will answer — confidently, with
one of your two labels and a probability. That property is not a bug in this notebook; it is
true of nearly every classifier deployed anywhere, and the confidence number does not warn you.

**Two things to do:**

1. **Out-of-domain.** Write a `predict(s)` helper around `model.predict_proba(vectorizer
   .transform([s]))[0, 1]` and feed it three inputs it has no business classifying. Note the
   probabilities. Anything near 0.5 means "no evidence either way" — the honest answer — but
   watch how often you get 0.8 on a sentence about nothing.
2. **A real error.** Find a document from the held-out half that it gets wrong, print it in
   full, and read it. Why did it fail? Is the document ambiguous even to you, is it short, or
   did one strong word drag it across?

That second move is Underwood's Pynchon misread, done on your own corpus. His genre classifier
called *The Crying of Lot 49* detective fiction, and the mistake is the most-cited thing about
the model, because it showed the boundary was fuzzy in a way the accuracy number never could.
**A classifier's errors are where it tells you what your categories actually are.**

In [ ]:
# 1. def predict(s): ... -> print the label and the probability.
# 2. Three out-of-domain inputs (Shakespeare, a recipe, another language).
# 3. One real held-out document it gets wrong, printed in full.
#
# WHAT IT SAID ABOUT THINGS IT COULDN'T KNOW: ...
# THE REAL ERROR, AND WHY WE THINK IT HAPPENED: ...

---

## Station 7 · Change one thing, fit again (6 min) — the point of the session

One model is a result. Several models are an argument. Fit at least three more, changing
**exactly one** decision at a time, and keep the numbers side by side — that discipline is the
difference between modelling and having built a thing that runs.

| Dial | Change | What it does, and why you'd want it |
|---|---|---|
| **Rare words** | `CountVectorizer(min_df=3)` | A word must appear in at least 3 documents to get a column. Drops thousands of one-off spellings, names, and typos that let the model memorise individual documents. |
| **Weighting** | `TfidfVectorizer()` | Counts, discounted by how many documents a word appears in. Week 2's tool: common-everywhere words stop shouting. |
| **Word pairs** | `ngram_range=(1, 2)` | Adds two-word features, so *this sub* and *new york* become single columns instead of four unrelated ones. Multiplies the vocabulary; pair it with `min_df`. |
| **Regularisation** | `LogisticRegression(C=0.1)` vs `C=10` | How hard the model is pushed toward small weights. Low C = simpler, fewer big claims, usually generalises better. High C = it trusts your training data more, and overfits it faster. |
| **Imbalance** | `class_weight="balanced"` | Makes errors on the smaller pile cost more, so the model stops riding whichever side is bigger. |

**Build a comparison table.** One row per model: what you changed, the held-out accuracy, and
the top three words per side. `results = []`, append a dict per fit, `pd.DataFrame(results)` at
the end. Keep `random_state=0` everywhere so you're comparing models, not splits.

**Two questions the table has to answer**, and they are the ones you'll be asked at report-back:

1. **Does the accuracy actually move?** If four quite different models all land within a point
   of each other — and, if you ran it, within the cross-validation spread from Station 4 — then
   your modelling choices weren't what determined the answer. The corpus was.
2. **Do the top words change more than the accuracy does?** This is the common and unsettling
   result: the same score, a visibly different explanation. It means the score was never the
   finding, and anyone reporting only the accuracy would have hidden the disagreement.

**Then commit.** Which model would you put in an essay, and what is the honest sentence for why
— not "it scored highest" but why its choices suit the question you wrote at Station 1?

In [ ]:
# results = []
#
# For each variant (change ONE thing, keep everything else fixed):
#   - re-vectorize if the change is to the features, re-split, refit
#   - results.append({"change": "min_df=3", "accuracy": ..., "top_A": ..., "top_B": ...})
#
# pd.DataFrame(results)
#
# DOES THE ACCURACY MOVE: ...
# DO THE TOP WORDS MOVE MORE THAN THE ACCURACY: ...
# THE MODEL WE'D KEEP, AND WHY: ...

---

## Station 8 · Report back (3 min)

Ninety seconds a group, and the ten answers together are the actual finding of the session.
Fill this in, in your own copy, before the clock runs out.

- **Our two piles:** \_\_\_ vs \_\_\_ , chosen because \_\_\_
- **Baseline / our best model:** \_\_\_ % / \_\_\_ %
- **The model we kept, and the one decision that made it best:** \_\_\_
- **Top words, side A:** \_\_\_ **side B:** \_\_\_
- **Topic, register, or habit?** \_\_\_
- **Where it broke:** \_\_\_
- **One caveat we would put in writing:** \_\_\_
- **What we would need to believe this:** \_\_\_

The last two are the ones the room will actually argue about. A caveat is not a disclaimer:
it's the specific thing that would have to be checked before your sentence is safe to publish
("the top words may be one prolific user; we didn't check per-author").

In [ ]:
# Optional: save today's work so it survives the runtime.
# - the results table to a CSV in PROJECT_DIR
# - the top-weights bar chart as a PNG
# Both are Week 4-quality artifacts if your project ends up using classification.

---

### If you finish early

- **A third class.** Add a pile and refit. Logistic regression handles it (one set of weights
  per class), `classification_report` gets more interesting, and the confusion matrix starts
  telling a story about which pair the model can't separate.
- **How much data do you actually need?** Refit on 50, 100, 200, 400 rows per side and plot
  accuracy against corpus size. Most curves flatten sooner than people expect, which is a
  useful thing to know before you spend Week 4 collecting.
- **Whose words are they?** If one prolific author wrote a tenth of one pile, the model may
  have learned that person rather than that community. Group by author and refit without the
  loudest few.
- **The same pair, a year apart.** Pull each side from two different time windows and compare
  the weights. Communities drift; that's a Week 2 trend question with a Week 3 tool.

---

### Cheat sheet

| You want | Reach for |
|---|---|
| text → matrix | `CountVectorizer()`, `TfidfVectorizer()`, `.fit_transform(texts)` |
| the column names | `vectorizer.get_feature_names_out()` |
| hold data out | `train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)` |
| fit | `LogisticRegression(max_iter=1000).fit(Xtr, ytr)` |
| score | `model.score(Xte, yte)` or `accuracy_score(yte, model.predict(Xte))` |
| the floor | `DummyClassifier(strategy="most_frequent")` |
| which errors | `confusion_matrix(yte, preds)`, `classification_report(yte, preds)` |
| how lucky was that | `cross_val_score(model, X, y, cv=5)` |
| the weights | `model.coef_.ravel()`, then `.argsort()` |
| a probability | `model.predict_proba(vectorizer.transform([s]))[0, 1]` |

**Stuck?** Ask the AI for the piece, not the notebook. Predict, run, interrogate — every cell,
all term. The worked version of this pipeline is `week03_classification.ipynb`; errors are in
`../kits/common-errors-cheatsheet.md`.

### The homework this feeds

Your sketch is one of today's models on a labelled set *you* are curious about, plus a
screenshot of its five most positive and five most negative words — do they make sense? And
bring your **corpus existence proof** to Week 4: a screenshot of 50 loadable rows of the data
you want to use. No proof, no pitch.